# 11 — 3D raw image workflow with placeholder calibration and automatic bins

This notebook is for the case where you have **only 3D FPD raw image layers** and no proper calibrated mass spectrum CSV.

Workflow:

```text
raw 3D image layers
→ total counts vs detector channel
→ placeholder channel→mass calibration
→ construct pseudo-spectrum
→ automatic peak detection
→ tentative isotope assignment
→ mass-bin creation
→ convert bins to channel bins
→ build 3D ion volumes
→ plot projections, depth profiles, and layer slider
```

⚠️ **Important scientific warning**

The placeholder calibration is only a software/testing fallback.  
Any isotope/element assignment made from this calibration is **tentative and not publication-grade**.

Replace the placeholder calibration with a real instrument calibration as soon as possible.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from pymagsims import Spectrum, SIMSVolume
from pymagsims.raw_image import SIMSRawImage
from pymagsims.isotopes import load_builtin_isotopes
from pymagsims.plotting import plot_ion_image_grid, plot_volume_slice

DATA = Path("../data")
RAW_LAYER_DIR = DATA / "3d"

## 1. Load raw 3D image layer files with natural sorting

Normal string sorting gives `Image_1, Image_10, Image_11, ..., Image_2`.

Natural sorting preserves the correct layer order.

In [ ]:
def natural_sort_key(path):
    return [
        int(text) if text.isdigit() else text.lower()
        for text in re.split(r"(\d+)", path.name)
    ]

paths = sorted(
    RAW_LAYER_DIR.glob("*Image_*.raw"),
    key=natural_sort_key,
)

print(f"Found {len(paths)} raw image layers")
for p in paths:
    print(p.name)

## 2. Build total counts vs detector channel

We sum all detected events across all raw image layers.

This gives a **channel-domain spectrum**:

```text
detector channel → total counts
```

In [ ]:
IMAGE_SHAPE = (256, 256)  # change to (512, 512) or other acquisition size if needed

all_counts = []

for path in paths:
    raw = SIMSRawImage.from_fpd_raw(path, shape=IMAGE_SHAPE)
    counts = raw.events["Channel"].value_counts()
    all_counts.append(counts)

channel_counts = (
    pd.concat(all_counts, axis=1)
    .fillna(0)
    .sum(axis=1)
    .sort_index()
)

channel_spectrum = pd.DataFrame(
    {
        "Channel": channel_counts.index.astype(int),
        "Counts": channel_counts.values.astype(int),
    }
)

display(channel_spectrum.head())
display(channel_spectrum.tail())

print("Channel range:", channel_spectrum["Channel"].min(), "to", channel_spectrum["Channel"].max())
print("Total counts:", channel_spectrum["Counts"].sum())

## 3. Plot total counts vs detector channel interactively

Use this plot to inspect peaks and sanity-check the channel-domain data.

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=channel_spectrum["Channel"],
        y=channel_spectrum["Counts"],
        mode="lines",
        name="Total counts",
        hovertemplate="Channel: %{x}<br>Counts: %{y}<extra></extra>",
    )
)

fig.update_layout(
    title="Total counts vs detector channel",
    xaxis_title="Detector channel",
    yaxis_title="Total counts",
    template="plotly_white",
    hovermode="closest",
)

fig.update_yaxes(type="log")

fig

## 4. Create a placeholder channel→mass calibration

This creates a linear placeholder calibration.

The values below are based on earlier example files:

```text
channel 1     → mass ≈ 1.700559 amu
channel 12000 → mass ≈ 126.608148 amu
```

⚠️ If your acquisition differs, change these values or load a real calibration CSV.

In [ ]:
calibration_dir = DATA / "calibration"
calibration_dir.mkdir(parents=True, exist_ok=True)

calibration_file = calibration_dir / "placeholder_channel_mass_calibration.csv"

n_channels = 12000
channels = np.arange(1, n_channels + 1)

mass_min = 1
mass_max = 260

masses = np.linspace(mass_min, mass_max, n_channels)

calibration_df = pd.DataFrame(
    {
        "Channel": channels,
        "Mass": masses,
    }
)

calibration_df.to_csv(calibration_file, index=False)

print(f"Wrote placeholder calibration to: {calibration_file}")
display(calibration_df.head())
display(calibration_df.tail())

## 5. Construct a placeholder-calibrated Spectrum object

We merge the channel counts with the placeholder mass calibration.

This creates a `Spectrum` object so we can reuse the existing peak detection,
isotope assignment, and bin creation tools.

In [ ]:
pseudo_spectrum_df = calibration_df.merge(
    channel_spectrum.rename(columns={"Counts": "Amplitude"}),
    on="Channel",
    how="left",
)

pseudo_spectrum_df["Amplitude"] = pseudo_spectrum_df["Amplitude"].fillna(0)

spec = Spectrum(
    data=pseudo_spectrum_df[["Channel", "Mass", "Amplitude"]],
    metadata={
        "source": "3D raw image layers with placeholder calibration",
        "warning": "Placeholder calibration only; isotope assignments are tentative.",
    },
    name="placeholder_calibrated_3d_raw_image_spectrum",
)

spec.plot(x="Mass", y="Amplitude", log_y=True);

## 6. Automatic peak detection and tentative isotope assignment

Because the mass calibration is a placeholder, assignments are only useful for software testing
or rough exploration.

In [ ]:
isotopes = load_builtin_isotopes()

assignments = spec.assign_peaks(
    isotope_table=isotopes,
    tolerance=0.3,
    prominence=50,
    distance=10,
)

display(assignments)
print(f"Number of candidate assignments: {len(assignments)}")

## 7. Plot pseudo-spectrum with assigned peaks

In [ ]:
fig, ax, assignments = spec.plot_with_peaks(
    isotope_table=isotopes,
    tolerance=0.3,
    prominence=50,
    distance=10,
    log_y=True,
    annotate=True,
)

In [ ]:
fig, assignments = spec.plot_with_peaks_interactive(
    isotope_table=isotopes,
    tolerance=0.3,
    prominence=50,
    distance=10,
    log_y=True,
    label_peaks=True,
)

fig

## 8. Create mass bins from the tentative assignments

These bins are mass-domain bins first.

Next we convert them to detector-channel bins before applying them to raw image layers.

In [ ]:
mass_bins = spec.create_bins_from_assignments(
    assignments,
    width=0.1,
)

display(mass_bins)

## 9. Save mass bins for later reuse

You can reuse this file in later notebooks.

Again, remember that these bins are based on a placeholder calibration unless replaced.

In [ ]:
bin_dir = DATA / "bins"
bin_dir.mkdir(parents=True, exist_ok=True)

mass_bin_file = bin_dir / "placeholder_mass_bins.csv"
mass_bins.to_csv(mass_bin_file, index=False)

print(f"Saved mass bins to: {mass_bin_file}")

## 10. Convert mass bins to channel bins

Raw image files contain detector channels, not masses.

Therefore the bins must contain:

```text
label, ch_min, ch_max
```

In [ ]:
def mass_to_channel_from_calibration(calibration_df, mass):
    idx = (calibration_df["Mass"] - mass).abs().idxmin()
    return int(calibration_df.loc[idx, "Channel"])


def mass_bins_to_channel_bins(mass_bins, calibration_df):
    converted = mass_bins.copy()

    converted["ch_min"] = converted["mass_min"].apply(
        lambda m: mass_to_channel_from_calibration(calibration_df, m)
    )
    converted["ch_max"] = converted["mass_max"].apply(
        lambda m: mass_to_channel_from_calibration(calibration_df, m)
    )

    ch_low = converted[["ch_min", "ch_max"]].min(axis=1)
    ch_high = converted[["ch_min", "ch_max"]].max(axis=1)

    converted["ch_min"] = ch_low.astype(int)
    converted["ch_max"] = ch_high.astype(int)

    return converted

In [ ]:
converted_bins = mass_bins_to_channel_bins(
    mass_bins,
    calibration_df,
)

display(converted_bins.head(20))

channel_bin_file = bin_dir / "placeholder_converted_channel_bins.csv"
converted_bins.to_csv(channel_bin_file, index=False)

print(f"Saved converted channel bins to: {channel_bin_file}")

## 11. Select bins for 3D reconstruction

To keep reconstruction fast, start with a small subset.

You can later select specific labels, for example:

```python
selected_bins = converted_bins[converted_bins["label"].isin(["28Si", "69Ga"])]
```

In [ ]:
selected_bins = converted_bins.head(10).copy()

display(selected_bins[["label", "mass_min", "mass_max", "ch_min", "ch_max"]])

## 12. Build 3D ion volumes from raw image layers

This creates one 3D array per bin/ion label.

Each array has shape:

```text
(z, y, x)
```

In [ ]:
volume = SIMSVolume.from_fpd_raw_image_series(
    paths=paths,
    bins=selected_bins,
    spectrum=None,
    include_total=True,
    shape=IMAGE_SHAPE,
)

print(volume.labels())
display(volume.metadata)

for label, arr in volume.volumes.items():
    print(f"{label}: shape={arr.shape}, total_counts={arr.sum()}")

## 13. Plot one layer

In [ ]:
plot_volume_slice(
    volume,
    label="Total",
    z=0,
    log=True,
    cmap="viridis",
);

In [ ]:
label = [l for l in volume.labels() if l != "Total"][0]

plot_volume_slice(
    volume,
    label=label,
    z=0,
    log=True,
    cmap="magma",
);

## 14. Plot summed projections

Summed projections collapse the 3D volume along the depth axis.

In [ ]:
projection_images = {
    label: volume.sum_projection(label)
    for label in volume.labels()
}

plot_ion_image_grid(
    projection_images,
    log=True,
    ncols=3,
    cmaps=["gray", "viridis", "magma", "plasma", "cividis", "inferno", "turbo"],
);

## 15. Depth profiles from the 3D volume

This integrates X/Y counts for each layer.

In [ ]:
profiles = []

for label in volume.labels():
    profile = volume.depth_profile(label)
    profile = profile.rename(columns={label: "Intensity"})
    profile["Label"] = label
    profiles.append(profile)

depth_profiles = pd.concat(profiles, ignore_index=True)

display(depth_profiles.head())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

for label, group in depth_profiles.groupby("Label"):
    ax.plot(
        group["Slice"],
        group["Intensity"],
        marker="o",
        linewidth=1,
        label=label,
    )

ax.set_xlabel("Slice / layer")
ax.set_ylabel("Integrated counts")
ax.set_yscale("log")
ax.grid(True)
ax.legend()

fig.tight_layout()

## 16. Plotly layer slider

In [ ]:
def plot_volume_slider(volume, label="Total", log=True, colorscale="Viridis"):
    arr = volume.get(label)
    arr_plot = np.log1p(arr) if log else arr

    z_count = arr_plot.shape[0]

    fig = go.Figure()

    for z in range(z_count):
        fig.add_trace(
            go.Heatmap(
                z=arr_plot[z],
                colorscale=colorscale,
                visible=(z == 0),
                colorbar=dict(title="log(1 + counts)" if log else "counts"),
            )
        )

    steps = []
    for z in range(z_count):
        steps.append(
            dict(
                method="update",
                args=[
                    {"visible": [i == z for i in range(z_count)]},
                    {"title": f"{label} — layer {z}"}
                ],
                label=str(z),
            )
        )

    fig.update_layout(
        title=f"{label} — layer 0",
        xaxis_title="X pixel",
        yaxis_title="Y pixel",
        yaxis=dict(scaleanchor="x", autorange="reversed"),
        width=700,
        height=700,
        sliders=[
            dict(
                active=0,
                currentvalue={"prefix": "Layer: "},
                pad={"t": 50},
                steps=steps,
            )
        ],
    )

    return fig

In [ ]:
plot_volume_slider(volume, label="102Rb", log=True)

In [ ]:
# Plot first non-total bin with slider
label = [l for l in volume.labels() if l != "Total"][7]
plot_volume_slider(volume, label=label, log=True, colorscale="Magma")

## Summary

This notebook enables a complete software test without a real mass spectrum CSV:

```text
raw 3D image layers
→ placeholder calibration
→ pseudo-spectrum
→ peak detection
→ tentative assignment
→ bins
→ 3D ion volumes
```

For real scientific interpretation, replace the placeholder calibration with a real channel→mass calibration.